# Demo 4: Adatbiztonság

**Kapcsolódó diák:** 30–37 (CIA-triád, GDPR, titkosítás, maszkolás, RBAC)

**Előfeltétel:** `docker compose up -d`  
JupyterLab: http://localhost:8888 (token: `demo`)  
PostgreSQL: `localhost:5432` (user: labor, pass: labor, db: week06)

Tartalom:
1. Column encryption – Fernet (AES-128-CBC)
2. PII maszkolás és pszeudoanonimizálás
3. RBAC – szerepkör alapú hozzáférés DuckDB-vel
4. Audit log implementáció


In [1]:
!pip install -q pandas cryptography duckdb sqlalchemy psycopg2-binary

## 1. Column encryption – Fernet

A Fernet AES-128-CBC + HMAC-SHA256 titkosítást alkalmaz. Valós rendszerben
a kulcsot KMS (AWS KMS, HashiCorp Vault) tárolja.


In [2]:
from cryptography.fernet import Fernet
import base64, hashlib
import pandas as pd

def generate_key(passphrase: str) -> bytes:
    key = hashlib.sha256(passphrase.encode()).digest()
    return base64.urlsafe_b64encode(key)

class ColumnEncryptor:
    def __init__(self, passphrase: str):
        self.cipher = Fernet(generate_key(passphrase))

    def encrypt_column(self, series):
        return series.apply(
            lambda v: self.cipher.encrypt(str(v).encode()).decode() if pd.notna(v) else v
        )

    def decrypt_column(self, series):
        return series.apply(
            lambda v: self.cipher.decrypt(v.encode()).decode() if pd.notna(v) else v
        )


df = pd.DataFrame({
    'order_id':    [1, 2, 3],
    'customer_id': [101, 102, 103],
    'email':       ['kiss.jozsef@ceg.hu', 'nagy.maria@mail.com', 'toth.peter@example.org'],
    'amount':      [12500.0, 3200.0, 8750.0],
})

enc = ColumnEncryptor('titkos_kulcs_2024')
df_enc = df.copy()
df_enc['email'] = enc.encrypt_column(df['email'])

print('=== Titkosított DataFrame ===')
df_enc['email_preview'] = df_enc['email'].str[:30] + '...'
print(df_enc[['order_id', 'customer_id', 'amount', 'email_preview']].to_string(index=False))

df_dec = df_enc.copy()
df_dec['email'] = enc.decrypt_column(df_enc['email'])
print('\n=== Visszafejtett DataFrame ===')
print(df_dec[['order_id', 'customer_id', 'amount', 'email']].to_string(index=False))

=== Titkosított DataFrame ===
 order_id  customer_id  amount                     email_preview
        1          101 12500.0 gAAAAABp5zF5GXVbqzHScDpO8Z0c-F...
        2          102  3200.0 gAAAAABp5zF5e_uG9lgBlLYnsQdCaM...
        3          103  8750.0 gAAAAABp5zF5BtEijQRu7W4Vd5mgFu...

=== Visszafejtett DataFrame ===
 order_id  customer_id  amount                  email
        1          101 12500.0     kiss.jozsef@ceg.hu
        2          102  3200.0    nagy.maria@mail.com
        3          103  8750.0 toth.peter@example.org


## 2. PII maszkolás és pszeudoanonimizálás

A maszkolás visszafordíthatatlan – az anonimizált adat kiesik a GDPR hatálya alól.
A pszeudoanonimizálás visszafejthető (lookup tábla alapján), GDPR hatálya alatt marad.


In [3]:
import re
import hashlib
import pandas as pd

class DataMasker:
    @staticmethod
    def mask_email(email: str) -> str:
        if not email or '@' not in email: return email
        local, domain = email.split('@', 1)
        masked_local  = local[0] + '***'
        domain_parts  = domain.split('.')
        return f'{masked_local}@{domain_parts[0][0]}**.{domain_parts[-1]}'

    @staticmethod
    def mask_phone(phone: str) -> str:
        digits = re.sub(r'\D', '', phone)
        return '*' * (len(digits) - 4) + digits[-4:]

    @staticmethod
    def pseudonymize(value: str, salt: str = 'secret_salt') -> str:
        return 'uid_' + hashlib.sha256(f'{salt}{value}'.encode()).hexdigest()[:12]

    @staticmethod
    def generalize_age(age: int, bucket: int = 10) -> str:
        lo = (age // bucket) * bucket
        return f'{lo}-{lo + bucket - 1}'


df = pd.DataFrame({
    'name':  ['Kiss József', 'Nagy Mária', 'Tóth Péter'],
    'email': ['kiss.jozsef@ceg.hu', 'nagy.maria@mail.com', 'toth.peter@example.org'],
    'phone': ['+36201234567', '+36301234568', '+36701234569'],
    'age':   [27, 34, 51],
})

masker = DataMasker()
df_masked = df.copy()
df_masked['email'] = df['email'].apply(masker.mask_email)
df_masked['phone'] = df['phone'].apply(masker.mask_phone)
df_masked['name']  = df['name'].apply(masker.pseudonymize)
df_masked['age']   = df['age'].apply(masker.generalize_age)

print('=== Maszkolás / pszeudoanonimizálás eredménye ===')
print(df_masked.to_string(index=False))

=== Maszkolás / pszeudoanonimizálás eredménye ===
            name        email       phone   age
uid_78606c14e3a1  k***@c**.hu *******4567 20-29
uid_13edb03ccd72 n***@m**.com *******4568 30-39
uid_59622143411d t***@e**.org *******4569 50-59


## 3. RBAC – szerepkör alapú hozzáférés-vezérlés

Az `analyst` csak EU adatot lát, email nélkül.
A `manager` minden régiót lát, de email nélkül.
Az `admin` teljes hozzáféréssel rendelkezik.


In [4]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("""
    CREATE TABLE orders AS SELECT * FROM (VALUES
        (1, 101, 'EU',   12500.0, 'sales@ceg.hu'),
        (2, 102, 'US',    3200.0, 'b2b@ceg.hu'),
        (3, 103, 'EU',    8750.0, 'sales@ceg.hu'),
        (4, 104, 'APAC', 45000.0, 'apac@ceg.hu')
    ) AS t(order_id, customer_id, region, amount, contact_email)
""")

def query_as_role(role: str) -> pd.DataFrame:
    if role == 'analyst':
        return con.execute("""
            SELECT order_id, customer_id, region, amount, '***' AS contact_email
            FROM orders WHERE region = 'EU'
        """).df()
    elif role == 'manager':
        return con.execute("""
            SELECT order_id, customer_id, region, amount, '***' AS contact_email
            FROM orders
        """).df()
    else:
        return con.execute('SELECT * FROM orders').df()

for role in ['analyst', 'manager', 'admin']:
    print(f'\n=== {role.upper()} szerepkör ===')
    print(query_as_role(role).to_string(index=False))


=== ANALYST szerepkör ===
 order_id  customer_id region  amount contact_email
        1          101     EU 12500.0           ***
        3          103     EU  8750.0           ***

=== MANAGER szerepkör ===
 order_id  customer_id region  amount contact_email
        1          101     EU 12500.0           ***
        2          102     US  3200.0           ***
        3          103     EU  8750.0           ***
        4          104   APAC 45000.0           ***

=== ADMIN szerepkör ===
 order_id  customer_id region  amount contact_email
        1          101     EU 12500.0  sales@ceg.hu
        2          102     US  3200.0    b2b@ceg.hu
        3          103     EU  8750.0  sales@ceg.hu
        4          104   APAC 45000.0   apac@ceg.hu


## 4. Audit log

Az audit log rögzíti, hogy ki, mikor, mihez fért hozzá.
GDPR elszámoltathatóság (Art. 5(2)) kötelezi az adatkezelőket ennek vezetésére.


In [5]:
from datetime import datetime
import json

class AuditLogger:
    def __init__(self):
        self.log: list = []

    def record(self, user: str, action: str, resource: str,
               outcome: str = 'SUCCESS', details: str = ''):
        entry = {
            'timestamp': datetime.now().isoformat(),
            'user': user, 'action': action,
            'resource': resource, 'outcome': outcome,
            'details': details
        }
        self.log.append(entry)

    def print_log(self):
        print(f'{"Timestamp":23}  {"User":20}  {"Action":8}  {"Resource":25}  Outcome')
        print('-' * 90)
        for e in self.log:
            print(f'{e["timestamp"][:19]:23}  {e["user"]:20}  '
                  f'{e["action"]:8}  {e["resource"]:25}  {e["outcome"]}')


logger = AuditLogger()
logger.record('bi.analyst@ceg.hu',    'SELECT', 'gold.monthly_revenue',  'SUCCESS')
logger.record('ml.engineer@ceg.hu',   'SELECT', 'silver.orders',         'SUCCESS')
logger.record('unknown@external.com', 'SELECT', 'silver.orders',         'DENIED', 'Nincs jogosultság')
logger.record('admin@ceg.hu',         'DELETE', 'bronze.orders',         'SUCCESS', 'GDPR törlési kérelem #4521')
logger.record('bi.analyst@ceg.hu',    'EXPORT', 'gold.monthly_revenue',  'SUCCESS', '5 000 sor exportálva')

print('=== Audit Log ===')
logger.print_log()
print(f'\nFigyelmeztetés: {sum(1 for e in logger.log if e["outcome"] == "DENIED")} elutasított kérelem!')

=== Audit Log ===
Timestamp                User                  Action    Resource                   Outcome
------------------------------------------------------------------------------------------
2026-04-21T08:12:41      bi.analyst@ceg.hu     SELECT    gold.monthly_revenue       SUCCESS
2026-04-21T08:12:41      ml.engineer@ceg.hu    SELECT    silver.orders              SUCCESS
2026-04-21T08:12:41      unknown@external.com  SELECT    silver.orders              DENIED
2026-04-21T08:12:41      admin@ceg.hu          DELETE    bronze.orders              SUCCESS
2026-04-21T08:12:41      bi.analyst@ceg.hu     EXPORT    gold.monthly_revenue       SUCCESS

Figyelmeztetés: 1 elutasított kérelem!
